# T2D cohort: GLP-1 versus DPP-4

Run this notebook inside the All of Us Researcher Workbench with authorised data access. It queries GLP-1 and DPP-4 exposures, constructs the treatment index and condition histories. Run the 'Pretraining_cohort.ipynb' notebook to generate the necessary dataframes for running this notebook.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "src" / "model.py").is_file() and (path / "runnables").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the repository or one of its subdirectories.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.project_paths import SHARED_DATA_DIR, COHORT_DATA_DIR

import pandas
import os
import json
from pathlib import Path

OUTPUT_DIR = COHORT_DATA_DIR / "T2D"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _load_all_of_us_config():
    config_path = PROJECT_ROOT / ".all_of_us_config.json"
    if not config_path.exists():
        return {}
    return json.loads(config_path.read_text())

def _ensure_workspace_cdr(config):
    if "WORKSPACE_CDR" in os.environ:
        return

    workspace_cdr = config.get("WORKSPACE_CDR")
    cdr_project = config.get("CDR_VERSION_CLOUD_PROJECT")
    cdr_dataset = config.get("CDR_VERSION_BIGQUERY_DATASET")
    if workspace_cdr:
        os.environ["WORKSPACE_CDR"] = workspace_cdr
    elif cdr_project and cdr_dataset:
        os.environ["WORKSPACE_CDR"] = f"{cdr_project}.{cdr_dataset}"
    else:
        raise KeyError("WORKSPACE_CDR is not set and the All of Us config is missing CDR project/dataset values")

all_of_us_config = _load_all_of_us_config()
_ensure_workspace_cdr(all_of_us_config)
BIGQUERY_PROJECT = (
    os.environ.get("GOOGLE_CLOUD_PROJECT")
    or os.environ.get("GOOGLE_PROJECT")
    or all_of_us_config.get("BILLING_CLOUD_PROJECT")
)
BIGQUERY_LOCATION = os.environ.get("BIGQUERY_LOCATION") or all_of_us_config.get("BIGQUERY_LOCATION") or "us-central1"
USE_BQSTORAGE_API = os.environ.get("BIGQUERY_STORAGE_API_ENABLED", "1") != "0"

import sys


from IPython.display import display


In [ ]:
import pandas as pd
conditions_df = pd.read_pickle((SHARED_DATA_DIR / 'events.pkl'))
patients_df = pd.read_pickle((SHARED_DATA_DIR / 'patients.pkl'))

In [ ]:
import pandas_gbq
# GLP-1 agonists
import pandas
import os

# This query represents dataset "findings trajectories" for domain "drug" and was generated for All of Us Registered Tier Dataset v8
dataset_81958204_drug_sql = """
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary,
        d_exposure.drug_exposure_start_datetime,
        d_exposure.drug_type_concept_id,
        d_type.concept_name as drug_type_concept_name,
        d_visit.concept_name as visit_occurrence_concept_name,
        d_exposure.drug_source_value,
        d_exposure.drug_source_concept_id,
        d_source_concept.concept_name as source_concept_name,
        d_source_concept.concept_code as source_concept_code,
        d_source_concept.vocabulary_id as source_vocabulary,
        d_exposure.route_source_value
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (SELECT
                    DISTINCT ca.descendant_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                JOIN
                    (SELECT
                        DISTINCT c.concept_id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id             
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                        WHERE
                            concept_id IN (1123618)             
                            AND full_text LIKE '%_rank1]%'       ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) b 
                        ON (ca.ancestor_id = b.concept_id)))  
                    AND (d_exposure.PERSON_ID IN (SELECT
                        distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) d_exposure 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_standard_concept 
            ON d_exposure.drug_concept_id = d_standard_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_type 
            ON d_exposure.drug_type_concept_id = d_type.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_route 
            ON d_exposure.route_concept_id = d_route.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
            ON d_exposure.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_visit 
            ON v.visit_concept_id = d_visit.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_source_concept 
            ON d_exposure.drug_source_concept_id = d_source_concept.concept_id"""

dataset_81958204_drug_df = pandas_gbq.read_gbq(
    dataset_81958204_drug_sql,
    dialect="standard",
    project_id=BIGQUERY_PROJECT,
    location=BIGQUERY_LOCATION,
    use_bqstorage_api=USE_BQSTORAGE_API,
    progress_bar_type="tqdm_notebook")

dataset_81958204_drug_df.head(5)
drugs_df = dataset_81958204_drug_df.copy()

In [ ]:
dataset_81958204_drug_df.to_pickle(OUTPUT_DIR / "glp1_drugs_df.pkl")

In [ ]:
import pandas_gbq
# DPP-4 Inhibitors
import pandas
import os

dataset_25053772_drug_sql = """
WITH drug_base AS (
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary,
        d_exposure.drug_exposure_start_datetime,
        d_exposure.drug_type_concept_id,
        d_type.concept_name as drug_type_concept_name,
        d_visit.concept_name as visit_occurrence_concept_name,
        d_exposure.drug_source_value,
        d_exposure.drug_source_concept_id,
        d_source_concept.concept_name as source_concept_name,
        d_source_concept.concept_code as source_concept_code,
        d_source_concept.vocabulary_id as source_vocabulary,
        d_exposure.route_source_value,
        d_exposure.dose_unit_source_value
    FROM
        (
            SELECT
                *
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.drug_exposure` d_exposure
            WHERE
                (
                    drug_concept_id IN (
                        SELECT DISTINCT
                            ca.descendant_id
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca
                        JOIN
                            (
                                SELECT DISTINCT
                                    c.concept_id
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c
                                JOIN
                                    (
                                        SELECT
                                            CAST(cr.id as string) AS id
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr
                                        WHERE
                                            concept_id IN (21600783)
                                            AND full_text LIKE '%_rank1]%'
                                    ) a
                                    ON (
                                        c.path LIKE CONCAT('%.', a.id, '.%')
                                        OR c.path LIKE CONCAT('%.', a.id)
                                        OR c.path LIKE CONCAT(a.id, '.%')
                                        OR c.path = a.id
                                    )
                                WHERE
                                    is_standard = 1
                                    AND is_selectable = 1
                            ) b
                            ON ca.ancestor_id = b.concept_id
                    )
                    AND d_exposure.person_id IN (
                        SELECT DISTINCT
                            cb_search_person.person_id
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person
                        WHERE
                            cb_search_person.person_id IN (
                                SELECT
                                    person_id
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p
                            )
                            AND cb_search_person.person_id IN (
                                SELECT
                                    person_id
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p
                                WHERE
                                    has_ehr_data = 1
                            )
                    )
                )
        ) d_exposure
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_standard_concept
        ON d_exposure.drug_concept_id = d_standard_concept.concept_id
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_type
        ON d_exposure.drug_type_concept_id = d_type.concept_id
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_route
        ON d_exposure.route_concept_id = d_route.concept_id
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v
        ON d_exposure.visit_occurrence_id = v.visit_occurrence_id
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_visit
        ON v.visit_concept_id = d_visit.concept_id
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_source_concept
        ON d_exposure.drug_source_concept_id = d_source_concept.concept_id
),

atc_rollup AS (
    SELECT
        ca.descendant_concept_id AS drug_concept_id,

        STRING_AGG(
            DISTINCT CASE
                WHEN c.concept_class_id = 'ATC 2nd'
                THEN CAST(c.concept_id AS STRING)
                ELSE NULL
            END,
            ' | '
        ) AS atc_level2_concept_ids,

        STRING_AGG(
            DISTINCT CASE
                WHEN c.concept_class_id = 'ATC 2nd'
                THEN c.concept_name
                ELSE NULL
            END,
            ' | '
        ) AS atc_level2_names,

        STRING_AGG(
            DISTINCT CASE
                WHEN c.concept_class_id = 'ATC 2nd'
                THEN c.concept_code
                ELSE NULL
            END,
            ' | '
        ) AS atc_level2_codes,

        STRING_AGG(
            DISTINCT CASE
                WHEN c.concept_class_id = 'ATC 3rd'
                THEN CAST(c.concept_id AS STRING)
                ELSE NULL
            END,
            ' | '
        ) AS atc_level3_concept_ids,

        STRING_AGG(
            DISTINCT CASE
                WHEN c.concept_class_id = 'ATC 3rd'
                THEN c.concept_name
                ELSE NULL
            END,
            ' | '
        ) AS atc_level3_names,

        STRING_AGG(
            DISTINCT CASE
                WHEN c.concept_class_id = 'ATC 3rd'
                THEN c.concept_code
                ELSE NULL
            END,
            ' | '
        ) AS atc_level3_codes,

        STRING_AGG(
            DISTINCT CASE
                WHEN c.concept_class_id = 'ATC 4th'
                THEN CAST(c.concept_id AS STRING)
                ELSE NULL
            END,
            ' | '
        ) AS atc_level4_concept_ids,

        STRING_AGG(
            DISTINCT CASE
                WHEN c.concept_class_id = 'ATC 4th'
                THEN c.concept_name
                ELSE NULL
            END,
            ' | '
        ) AS atc_level4_names,

        STRING_AGG(
            DISTINCT CASE
                WHEN c.concept_class_id = 'ATC 4th'
                THEN c.concept_code
                ELSE NULL
            END,
            ' | '
        ) AS atc_level4_codes

    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.concept_ancestor` ca
    JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c
        ON ca.ancestor_concept_id = c.concept_id
    WHERE
        c.vocabulary_id = 'ATC'
        AND c.concept_class_id IN ('ATC 2nd', 'ATC 3rd', 'ATC 4th')
    GROUP BY
        ca.descendant_concept_id
)

SELECT
    db.*,
    ar.atc_level2_concept_ids,
    ar.atc_level2_names,
    ar.atc_level2_codes,
    ar.atc_level3_concept_ids,
    ar.atc_level3_names,
    ar.atc_level3_codes,
    ar.atc_level4_concept_ids,
    ar.atc_level4_names,
    ar.atc_level4_codes
FROM
    drug_base db
LEFT JOIN
    atc_rollup ar
    ON db.drug_concept_id = ar.drug_concept_id
"""

dataset_25053772_drug_df = pandas_gbq.read_gbq(
    dataset_25053772_drug_sql,
    dialect="standard",
    project_id=BIGQUERY_PROJECT,
    location=BIGQUERY_LOCATION,
    use_bqstorage_api=USE_BQSTORAGE_API,
    progress_bar_type="tqdm_notebook"
)

dataset_25053772_drug_df.head(5)

In [ ]:
dataset_25053772_drug_df.to_pickle(OUTPUT_DIR / "dpp4_drugs_df.pkl")

## Select eligible GLP-1 and DPP-4 patients

The next cells derive treatment index dates and condition-history coverage from the source tables, then apply the baseline and follow-up requirements configured below. `patient_timeline` supports eligibility and attrition checks; `eligible_cohort` contains the final selected patients.


In [ ]:
# Cohort eligibility: minimum condition-history coverage around index treatment.
BASELINE_DAYS_MIN = 365
FOLLOWUP_DAYS_MIN = 0


In [ ]:
import numpy as np

condition_dates = conditions_df.copy()
condition_dates["condition_start_datetime"] = pandas.to_datetime(
    condition_dates["condition_start_datetime"],
    utc=True,
    errors="coerce",
)
condition_dates["icd_level3"] = condition_dates["icd_level3_code"].fillna(
    condition_dates["condition_source_value"].astype(str).str.split(".").str[0]
)
condition_dates = condition_dates.dropna(subset=["condition_start_datetime"])

glp_exposures = drugs_df.copy()
glp_exposures["drug_exposure_start_datetime"] = pandas.to_datetime(
    glp_exposures["drug_exposure_start_datetime"],
    utc=True,
    errors="coerce",
)
glp_exposures["study_drug_class"] = "glp1"

dpp4_exposures = dataset_25053772_drug_df.copy()
dpp4_exposures["drug_exposure_start_datetime"] = pandas.to_datetime(
    dpp4_exposures["drug_exposure_start_datetime"],
    utc=True,
    errors="coerce",
)
dpp4_exposures["study_drug_class"] = "dpp4"

study_exposures = pandas.concat([glp_exposures, dpp4_exposures], ignore_index=True)


def first_person_date(frame, date_col):
    return frame.groupby("person_id")[date_col].min()


def last_person_date(frame, date_col):
    return frame.groupby("person_id")[date_col].max()


first_condition_date = first_person_date(condition_dates, "condition_start_datetime")
last_condition_date = last_person_date(condition_dates, "condition_start_datetime")

first_t2d_date = first_person_date(
    condition_dates.loc[condition_dates["icd_level3"] == "E11"],
    "condition_start_datetime",
)
first_hf_date = first_person_date(
    condition_dates.loc[condition_dates["icd_level3"] == "I50"],
    "condition_start_datetime",
)
first_mi_date = first_person_date(
    condition_dates.loc[condition_dates["icd_level3"] == "I21"],
    "condition_start_datetime",
)

first_glp_date = first_person_date(glp_exposures, "drug_exposure_start_datetime")
first_dpp4_date = first_person_date(dpp4_exposures, "drug_exposure_start_datetime")
first_comparator_date = first_dpp4_date

last_glp_date = last_person_date(glp_exposures, "drug_exposure_start_datetime")
last_dpp4_date = last_person_date(dpp4_exposures, "drug_exposure_start_datetime")
last_study_treatment_date = last_person_date(study_exposures, "drug_exposure_start_datetime")

last_record_date = pandas.concat(
    [
        last_condition_date.rename("last_condition_date"),
        last_glp_date.rename("last_glp_date"),
        last_dpp4_date.rename("last_dpp4_date"),
    ],
    axis=1,
).max(axis=1)

patient_timeline = patients_df[["person_id"]].drop_duplicates().set_index("person_id")
patient_timeline = patient_timeline.join(first_condition_date.rename("first_condition_date"))
patient_timeline = patient_timeline.join(last_condition_date.rename("last_condition_date"))
patient_timeline = patient_timeline.join(first_t2d_date.rename("first_t2d_date"))
patient_timeline = patient_timeline.join(first_glp_date.rename("first_glp_date"))
patient_timeline = patient_timeline.join(first_dpp4_date.rename("first_dpp4_date"))
patient_timeline = patient_timeline.join(first_comparator_date.rename("first_comparator_date"))
patient_timeline = patient_timeline.join(last_glp_date.rename("last_glp_date"))
patient_timeline = patient_timeline.join(last_dpp4_date.rename("last_dpp4_date"))
patient_timeline = patient_timeline.join(last_study_treatment_date.rename("last_study_treatment_date"))
patient_timeline = patient_timeline.join(first_hf_date.rename("first_hf_date"))
patient_timeline = patient_timeline.join(first_mi_date.rename("first_mi_date"))
patient_timeline = patient_timeline.join(last_record_date.rename("last_record_date"))

patient_timeline["index_treatment_date"] = patient_timeline[
    ["first_glp_date", "first_dpp4_date"]
].min(axis=1)
patient_timeline["treatment_group"] = "untreated"
patient_timeline["index_treatment_class"] = "none"

same_day_cross_arm_start = (
    patient_timeline["first_glp_date"].notna()
    & patient_timeline["first_comparator_date"].notna()
    & (patient_timeline["first_glp_date"] == patient_timeline["first_comparator_date"])
)
glp_first = patient_timeline["first_glp_date"].notna() & (
    patient_timeline["first_comparator_date"].isna()
    | (patient_timeline["first_glp_date"] < patient_timeline["first_comparator_date"])
)
comparator_first = patient_timeline["first_comparator_date"].notna() & (
    patient_timeline["first_glp_date"].isna()
    | (patient_timeline["first_comparator_date"] < patient_timeline["first_glp_date"])
)

patient_timeline.loc[glp_first, "treatment_group"] = "glp1_initiator"
patient_timeline.loc[comparator_first, "treatment_group"] = "dpp4_initiator"
patient_timeline.loc[same_day_cross_arm_start, "treatment_group"] = "same_day_cross_arm_start"
patient_timeline.loc[glp_first, "index_treatment_class"] = "glp1"

patient_timeline.loc[comparator_first, "index_treatment_class"] = "dpp4"

later_glp = patient_timeline["first_glp_date"] > patient_timeline["index_treatment_date"]
later_dpp4 = patient_timeline["first_dpp4_date"] > patient_timeline["index_treatment_date"]

patient_timeline["switches_treatment_class"] = (
    (glp_first & later_dpp4) | (comparator_first & later_glp)
)

patient_timeline["days_from_t2d_to_treatment"] = (
    patient_timeline["index_treatment_date"] - patient_timeline["first_t2d_date"]
).dt.days
patient_timeline["condition_baseline_days"] = (
    patient_timeline["index_treatment_date"] - patient_timeline["first_condition_date"]
).dt.days
patient_timeline["condition_followup_days"] = (
    patient_timeline["last_condition_date"] - patient_timeline["index_treatment_date"]
).dt.days
patient_timeline["overall_followup_days_after_index"] = (
    patient_timeline["last_record_date"] - patient_timeline["index_treatment_date"]
).dt.days
patient_timeline["followup_days_after_index"] = patient_timeline["condition_followup_days"]

patient_timeline["days_from_index_to_hf"] = (
    patient_timeline["first_hf_date"] - patient_timeline["index_treatment_date"]
).dt.days
patient_timeline["days_from_index_to_mi"] = (
    patient_timeline["first_mi_date"] - patient_timeline["index_treatment_date"]
).dt.days
patient_timeline["days_from_last_treatment_to_hf"] = (
    patient_timeline["first_hf_date"] - patient_timeline["last_study_treatment_date"]
).dt.days
patient_timeline["days_from_last_treatment_to_mi"] = (
    patient_timeline["first_mi_date"] - patient_timeline["last_study_treatment_date"]
).dt.days

patient_timeline["days_to_hf"] = patient_timeline["days_from_index_to_hf"]
patient_timeline["days_to_mi"] = patient_timeline["days_from_index_to_mi"]


patient_timeline["has_t2d"] = patient_timeline["first_t2d_date"].notna()
patient_timeline["has_clear_treatment_arm"] = patient_timeline["treatment_group"].isin(
    ["glp1_initiator", "dpp4_initiator"]
)
patient_timeline["treatment_after_t2d"] = patient_timeline["days_from_t2d_to_treatment"] >= 0
patient_timeline["trajectory_candidate"] = (
    patient_timeline["has_t2d"]
    & patient_timeline["has_clear_treatment_arm"]
    & patient_timeline["treatment_after_t2d"]
)
patient_timeline["baseline_window_ok"] = patient_timeline["condition_baseline_days"] >= BASELINE_DAYS_MIN
patient_timeline["followup_window_ok"] = patient_timeline["condition_followup_days"] >= FOLLOWUP_DAYS_MIN
patient_timeline["base_candidate"] = (
    patient_timeline["trajectory_candidate"]
    & patient_timeline["baseline_window_ok"]
    & patient_timeline["followup_window_ok"]
)

for outcome_name in ["hf", "mi"]:
    patient_timeline[f"{outcome_name}_after_index"] = patient_timeline[f"days_from_index_to_{outcome_name}"] > 0
    patient_timeline[f"{outcome_name}_after_last_treatment"] = (
        patient_timeline[f"days_from_last_treatment_to_{outcome_name}"] > 0
    )
    patient_timeline[f"no_prior_{outcome_name}"] = (
        patient_timeline[f"days_from_index_to_{outcome_name}"].isna()
        | (patient_timeline[f"days_from_index_to_{outcome_name}"] > 0)
    )
    patient_timeline[f"{outcome_name}_candidate"] = (
        patient_timeline["base_candidate"] & patient_timeline[f"no_prior_{outcome_name}"]
    )

patient_timeline["days_from_index_to_last_treatment"] = (
    patient_timeline["last_study_treatment_date"] - patient_timeline["index_treatment_date"]
).dt.days

# Keep one patient-level table after applying all eligibility rules.
eligible_cohort = patient_timeline.loc[patient_timeline["base_candidate"]].reset_index().copy()
eligible_person_ids = eligible_cohort["person_id"].unique()


In [ ]:
def summarize_outcome_candidates(outcome_name, outcome_label):
    candidate_col = f"{outcome_name}_candidate"
    event_col = f"{outcome_name}_after_index"

    return (
        patient_timeline.loc[patient_timeline[candidate_col]]
        .groupby("treatment_group")
        .agg(
            patients=("treatment_group", "size"),
            events=(event_col, "sum"),
            event_rate=(event_col, "mean"),
            median_days_from_t2d_to_treatment=("days_from_t2d_to_treatment", "median"),
            median_condition_baseline_days=("condition_baseline_days", "median"),
            median_condition_followup_days=("condition_followup_days", "median"),
            switchers=("switches_treatment_class", "sum"),
        )
        .assign(outcome=outcome_label)
        .reset_index()
    )


def summarize_outcome_timing(outcome_name, outcome_label):
    event_col = f"{outcome_name}_after_index"
    after_last_col = f"{outcome_name}_after_last_treatment"
    days_from_index_col = f"days_from_index_to_{outcome_name}"
    days_from_last_col = f"days_from_last_treatment_to_{outcome_name}"

    event_frame = patient_timeline.loc[
        patient_timeline[f"{outcome_name}_candidate"] & patient_timeline[event_col]
    ]
    if event_frame.empty:
        return pandas.DataFrame(
            columns=[
                "treatment_group",
                "events",
                "events_after_last_treatment",
                "share_after_last_treatment",
                "median_days_from_index",
                "median_days_from_last_treatment",
                "outcome",
            ]
        )

    return (
        event_frame.groupby("treatment_group")
        .agg(
            events=("treatment_group", "size"),
            events_after_last_treatment=(after_last_col, "sum"),
            share_after_last_treatment=(after_last_col, "mean"),
            median_days_from_index=(days_from_index_col, "median"),
            median_days_from_last_treatment=(days_from_last_col, "median"),
        )
        .assign(outcome=outcome_label)
        .reset_index()
    )


attrition_masks = {
    "T2D diagnosis": patient_timeline["has_t2d"],
    "Clear GLP-1 vs DPP-4 arm": patient_timeline["has_t2d"] & patient_timeline["has_clear_treatment_arm"],
    "Treatment after T2D": patient_timeline["trajectory_candidate"],
    f"baseline >= {BASELINE_DAYS_MIN} days": patient_timeline["trajectory_candidate"] & patient_timeline["baseline_window_ok"],
    f"follow-up >= {FOLLOWUP_DAYS_MIN} days": patient_timeline["base_candidate"],
}

attrition_df = pandas.DataFrame(
    {
        "step": list(attrition_masks.keys()),
        "patients": [mask.sum() for mask in attrition_masks.values()],
    }
)

trajectory_summary = (
    patient_timeline.loc[patient_timeline["trajectory_candidate"]]
    .groupby("treatment_group")
    .agg(
        patients=("treatment_group", "size"),
        median_days_from_t2d_to_treatment=("days_from_t2d_to_treatment", "median"),
        median_condition_baseline_days=("condition_baseline_days", "median"),
        median_condition_followup_days=("condition_followup_days", "median"),
        median_days_from_index_to_last_treatment=("days_from_index_to_last_treatment", "median"),
    )
    .reset_index()
)

arm_summary = (
    patient_timeline.loc[patient_timeline["base_candidate"]]
    .groupby("treatment_group")
    .agg(
        patients=("treatment_group", "size"),
        median_days_from_t2d_to_treatment=("days_from_t2d_to_treatment", "median"),
        median_condition_baseline_days=("condition_baseline_days", "median"),
        median_condition_followup_days=("condition_followup_days", "median"),
        median_overall_followup_days=("overall_followup_days_after_index", "median"),
        switchers=("switches_treatment_class", "sum"),
    )
    .reset_index()
)

outcome_summary = pandas.concat(
    [
        summarize_outcome_candidates("hf", "Heart failure"),
        summarize_outcome_candidates("mi", "Acute myocardial infarction"),
    ],
    ignore_index=True,
)

outcome_timing_summary = pandas.concat(
    [
        summarize_outcome_timing("hf", "Heart failure"),
        summarize_outcome_timing("mi", "Acute myocardial infarction"),
    ],
    ignore_index=True,
)

print("Eligibility uses condition-based baseline and follow-up coverage around the treatment index.")
print("Eligible patients are those with T2D whose first observed study drug after diagnosis is GLP-1 or DPP-4.")
print("Baseline and follow-up windows are measured from the persisted condition trajectories.")
print()
display(attrition_df)
display(trajectory_summary)
display(arm_summary)
display(outcome_summary)
display(outcome_timing_summary)


In [ ]:
crossover_cohort = eligible_cohort.set_index("person_id").copy()

crossed_to_other_arm = crossover_cohort["switches_treatment_class"]

crossover_summary = (
    crossover_cohort.assign(received_other_study_drug_after_index=crossed_to_other_arm)
    .groupby("treatment_group")
    .agg(
        patients=("index_treatment_class", "size"),
        received_other_study_drug_after_index=("received_other_study_drug_after_index", "sum"),
    )
    .reindex(["glp1_initiator", "dpp4_initiator"], fill_value=0)
    .rename_axis("arm")
    .reset_index()
)
crossover_summary["received_other_study_drug_after_index_share"] = (
    crossover_summary["received_other_study_drug_after_index"]
    / crossover_summary["patients"].replace(0, np.nan)
).round(4)

print("Post-index GLP-1/DPP-4 crossover in the eligible cohort")
display(crossover_summary)

crossover_patients = crossover_cohort.loc[
    crossed_to_other_arm,
    [
        "treatment_group",
        "index_treatment_class",
        "index_treatment_date",
        "first_glp_date",
        "first_dpp4_date",
        "switches_treatment_class",
    ],
].reset_index()

display(crossover_patients.head(20))


## Export the eligible cohort event stream

`eligible_cohort` is the single patient-level table after treatment-arm assignment and baseline/follow-up eligibility filtering. Its `eligible_person_ids` select rows from the existing deduplicated event dataset. The event export preserves that dataset’s schema and includes each eligible patient’s recorded events; downstream treatment-effect preparation handles the index-relative history window. No additional patient-feature or nested trajectory tables are constructed here.


In [ ]:
events_df = pd.read_pickle((SHARED_DATA_DIR / 'events.pkl'))

In [ ]:
icd_conditions_treatment_after_t2d_df = events_df.loc[
    events_df["person_id"].isin(eligible_person_ids)
].copy()

print(f"Eligible cohort patients: {len(eligible_person_ids)}")
print(f"Event rows after eligibility filtering: {len(icd_conditions_treatment_after_t2d_df)}")
display(icd_conditions_treatment_after_t2d_df.head())


In [ ]:
icd_conditions_treatment_after_t2d_df.to_pickle(OUTPUT_DIR / "events.pkl")